# Incremental Bronze Ingestion — Batch 02

**Objective:** Ingest a new batch of fictional fintech transactions into an existing Bronze Delta table using PySpark, preserving raw records and validating the incremental ingestion results.

### Why?

**For our project:** Incorporates new transactions without modifying previously ingested records, preserving duplicates and invalid values for Silver validation.

**For Data Engineering:** Demonstrates incremental batch ingestion, Delta Lake append operations, data integrity checks, and transaction history for traceability.

#### 01 — Read and Validate Batch 02

Read the new CSV batch while preserving all fields as strings. Validate its six-column schema and eight source records to detect unexpected changes before ingestion.

In [0]:
# Read the new transaction batch without automatic type inference
df_batch_02 = spark.read.csv("/Volumes/workspace/default/fintech_landing/transactions_batch_02.csv", header=True, inferSchema=False)
display(df_batch_02)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX011,C001,2026-10-19 09:00:00,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19 10:00:00,CARD_PURCHASE,45.25,COMPLETED
TX013,C003,2026-10-19 11:00:00,BILL_PAYMENT,60.00,COMPLETED
TX014,C004,2026-10-19 12:00:00,PIX_OUT,30.00,PENDING
TX015,C005,2026-10-19 13:00:00,PIX_IN,900.00,FAILED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX016,C006,2026-10-19 14:00:00,TRANSFER,35.00,COMPLETED
TX017,C007,2026-10-19 15:00:00,PIX_IN,300.00,COMPLETED


In [0]:
# Verify the raw source column types
df_batch_02.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_ts: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
# Validate the expected number of incoming transactions
assert df_batch_02.count() == 8, "Unexpected Batch 2 file row count"

In [0]:
# Verify the expected source structure
assert len(df_batch_02.columns) == 6, "Unexpected number of columns for Batch 2 file"

#### 02 — Append New Transactions to Bronze

Verify the Bronze table state before Batch 02 ingestion and append the eight incoming records using Delta Lake. Append mode preserves existing data while adding the complete raw batch, including duplicates and invalid transactions.

In [0]:
# Read the Bronze snapshot before Batch 02 ingestion
bronze_table = "workspace.default.fintech_transactions_bronze"

df_bronze_before = (
    spark.read
    .option("versionAsOf", 2)
    .table(bronze_table)
)

display(df_bronze_before)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


In [0]:
# Verify the historical Bronze baseline before Batch 02
assert df_bronze_before.count() == 12, \
    "Unexpected initial Bronze row count"

In [0]:
# Append all raw Batch 02 records without modifying existing data
df_batch_02.write.format("delta").mode("append").saveAsTable(
    "workspace.default.fintech_transactions_bronze"
)

#### 03 — Verify Incremental Ingestion

Validate the Bronze table after Batch 02 ingestion and inspect Delta transaction history. Confirm that all eight source records were appended and that the original data was preserved.

In [0]:
# Read the Bronze snapshot immediately after Batch 02 ingestion
df_bronze_updated = (
    spark.read
    .option("versionAsOf", 3)
    .table(bronze_table)
)

display(df_bronze_updated)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


In [0]:
# Verify the Bronze total after appending eight transactions
assert df_bronze_updated.count() == 20, "Unexpected Updated Bronze table row count"

In [0]:
%sql
-- Verify the append operation and its recorded ingestion metrics
DESCRIBE HISTORY workspace.default.fintech_transactions_bronze;

#### Engineering Note — Ingestion Idempotency

This notebook demonstrates incremental ingestion using Delta Lake append mode. However, append mode does not track previously processed batches, so reprocessing the same batch can generate duplicate records.

Therefore, incremental ingestion does not necessarily guarantee idempotency.